# 영양성분 분석 코드

## 해당 코드는 재료의 영양 성분을 추출하는데 있음 

### 1. 라이브러리

In [7]:
import time  # 페이지 로딩 대기 및 서버 부하 방지를 위한 시간 지연
import json  # 수집된 데이터를 JSON 형식으로 변환 및 저장
import os    # 파일 경로 생성 및 디렉토리 확인을 위한 시스템 모듈
from selenium import webdriver  # 동적 웹페이지 제어를 위한 자동화 도구
from selenium.webdriver.chrome.options import Options  # 브라우저 환경 설정을 위한 옵션
from bs4 import BeautifulSoup  # HTML 데이터를 파싱하여 원하는 태그 추출
from concurrent.futures import ThreadPoolExecutor  # 병렬 처리를 통해 수집 속도 최적화
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

### 2. driver 셋팅 함수

In [8]:
# 브라우저 환경 최적화 함수
def get_safe_driver():
    # Chrome 브라우저 실행 옵션 설정
    chrome_options = Options()
    # 서버 환경이나 리소스 절약을 위해 샌드박스 및 GPU 비활성화
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--disable-dev-shm-usage")
    chrome_options.add_argument("--disable-gpu")
    # 이미지 로딩을 차단하여 페이지 로드 속도 획기적으로 개선
    chrome_options.add_argument("--blink-settings=imagesEnabled=false")
    # 페이지 로드 전략을 eager(DOM 완성 시점)로 설정하여 불필요한 대기 시간 제거
    chrome_options.page_load_strategy = 'eager'
    # 봇 차단을 방지하기 위한 사용자 에이전트 설정
    chrome_options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36")
    
    driver = webdriver.Chrome(options=chrome_options)
    return driver

### 3. FatSecret에서 재료의 영양 성분을 추출 함수

In [16]:
def get_nutrition_info(driver, ingredient_name):
    # 1. 검색 URL 설정
    encoded_query = ingredient_name.replace(" ", "%20")
    url = f"https://www.fatsecret.kr/%EC%B9%BC%EB%A1%9C%EB%A6%AC-%EC%98%81%EC%96%91%EC%86%8C/search?q={encoded_query}"
    driver.get(url)
    
    try:
        # 2. 링크 클릭 (위 코드 유지)
        wait = WebDriverWait(driver, 10)
        first_link = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "#content > table > tbody > tr > td.leftCell > div > table > tbody > tr:nth-child(1) > td > a")))
        first_link.click()
        time.sleep(2)
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')
    
        # 3. 안전한 추출 함수 정의
        def get_text_safe(selector):
            element = soup.select_one(selector)
            return element.text.strip() if element else "N/A"
        
        # 4. 정보 추출 (CSS 선택자 기반)
        nutrition_info = {
            "name": ingredient_name,
            "serving_size": get_text_safe("div.serving_size_value"),
            "kcal": get_text_safe("#content > table > tbody > tr > td.leftCell > div > table > tbody > tr > td.details > div > table:nth-child(2) > tbody > tr > td:nth-child(1) > div.factValue"),
            "carbs": get_text_safe("#content > table > tbody > tr > td.leftCell > div > table > tbody > tr > td.details > div > table:nth-child(2) > tbody > tr > td:nth-child(5) > div.factValue"),
            "protein": get_text_safe("#content > table > tbody > tr > td.leftCell > div > table > tbody > tr > td.details > div > table:nth-child(2) > tbody > tr > td:nth-child(7) > div.factValue"),
            "fat": get_text_safe("#content > table > tbody > tr > td.leftCell > div > table > tbody > tr > td.details > div > table:nth-child(2) > tbody > tr > td:nth-child(3) > div.factValue"),
        }
        
        return nutrition_info

    except Exception as e:
        print(f"[실패] {ingredient_name} 추출 중 오류 발생: {e}")
        return None
    finally:
        driver.quit()

### 4. 수집된 데이터는 건너뛰는(캐싱) 함수

In [10]:
def load_existing_data(file_path="nutrition_db.json"):
    # 파일이 없으면 빈 딕셔너리 반환
    if not os.path.exists(file_path):
        return {}
    
    # 파일이 있지만 내용이 비어있으면 빈 딕셔너리 반환
    if os.path.getsize(file_path) == 0:
        return {}
        
    # 파일 읽기
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except json.JSONDecodeError:
            # 파일이 JSON 형식이 아니거나 깨졌을 경우 초기화
            print("파일 형식이 올바르지 않아 초기화합니다.")
            return {}

### 5. 데이터 json형식으로 저장 함수

In [11]:
def save_data(data, file_path="nutrition_db.json"):
    # 1. 기존 데이터 로드 (파일이 있으면)
    if os.path.exists(file_path):
        with open(file_path, "r", encoding="utf-8") as f:
            try:
                db = json.load(f)
            except json.JSONDecodeError:
                db = {}
    else:
        db = {}
    
    # 2. 새로운 데이터 업데이트 (재료명을 키로 사용)
    if data:
        db[data['name']] = data
        
    # 3. 파일로 저장
    with open(file_path, "w", encoding="utf-8") as f:
        json.dump(db, f, indent=4, ensure_ascii=False)
    
    print(f"[{data['name']}] 데이터가 {file_path}에 저장되었습니다.")

### 6. 병렬 작업 함수

In [14]:
def task_crawl_ingredient(ingredient):
    options = Options()
    # options.add_argument("--headless")
    driver = webdriver.Chrome(options=options)
    
    try:
        data = get_nutrition_info(driver, ingredient)
        if data:
            save_data(data) # 이제 안전하게 호출 가능!
            return f"[완료] {ingredient}"
    finally:
        driver.quit()
    return f"[실패] {ingredient}"

### 테스트를 위한 코드

In [19]:
# 테스트를 위한 코드
if __name__ == "__main__":
    ingredients = [
        "삼겹살", "돼지고기 목살", "불고기감 소고기", "닭가슴살", "새우", "오징어", "고등어",
        "양파", "당근", "대파", "마늘", "양배추", "오이", "콩나물", "애호박", "감자", "고구마",
        "느타리버섯", "팽이버섯", "당면", "계란", "두부",
        "김치", "고추장", "된장", "간장", "설탕", "참기름", "고춧가루",
        # [추가] 육류 및 단백질
        "베이컨", "훈제오리", "참치캔", "스팸", "소시지",
        
        # [추가] 유제품 및 치즈
        "우유", "요거트", "체다치즈", "모짜렐라 치즈", "버터",
        
        # [추가] 곡류 및 면류
        "쌀", "현미", "귀리", "파스타면", "밀가루",
        
        # [추가] 채소 및 과일류
        "토마토", "사과", "바나나", "브로콜리", "시금치", "상추",
        
        # [추가] 견과류 및 기타
        "아몬드", "호두", "올리브유", "케첩", "마요네즈",
        "햄버거", "치즈버거", "불고기버거", "감자튀김", 
        "피자", "후라이드 치킨", "양념치킨", "짜장면", "짬뽕", 
        "김밥", "떡볶이", "순대",
        "안심", "등심", "차돌박이", "항정살", "닭다리살", "오리고기", "양고기", "제육용 돼지고기",
    
        # [추가] 곡물 종류
        "백미", "잡곡", "흑미", "메밀", "보리", "귀리", "옥수수", "병아리콩", "검은콩",
        
        # [추가] 아이스크림 및 디저트류
        "바닐라 아이스크림", "초코 아이스크림", "딸기 아이스크림", "녹차 아이스크림", "소프트 아이스크림"
    ]
    
    ingredients += [
    # [추가] 해산물 및 어패류
    "연어", "참치", "낙지", "쭈꾸미", "홍합", "가리비", "꽃게", "멸치", "어묵",
    
    # [추가] 베이커리 및 간식류
    "식빵", "바게트", "베이글", "머핀", "크로와상", "도넛", "카스테라",
    
    # [추가] 다양한 채소 및 향신료 (요리의 맛을 결정)
    "깻잎", "부추", "미나리", "청경채", "가지", "파프리카", "피망", "생강", "후추", "식초",
    
    # [추가] 과일 및 과채류 확장
    "포도", "귤", "키위", "블루베리", "레몬", "수박", "복숭아", "배",
    
    # [추가] 소스 및 드레싱 (영양 분석에 중요)
    "굴소스", "쌈장", "머스타드", "스리라차", "발사믹 드레싱", "데리야끼 소스",
    
    # [추가] 기타 가공식품
    "라면", "칼국수", "당면", "곤약", "두유", "프로틴 파우더"
]
    # 여기서부터 들여쓰기를 정확히 맞춰야 합니다
    existing_db = load_existing_data()
    ingredients_to_crawl = [ing for ing in ingredients if ing not in existing_db]
    
    print(f"수집 대상 재료: {ingredients_to_crawl}")

    if ingredients_to_crawl:
        print("병렬 수집을 시작합니다...")
        with ThreadPoolExecutor(max_workers=3) as executor:
            results = list(executor.map(task_crawl_ingredient, ingredients_to_crawl))
        print("모든 작업이 완료되었습니다:", results)
    else:
        print("이미 모든 재료의 데이터가 존재합니다.")

수집 대상 재료: ['수박']
병렬 수집을 시작합니다...
[수박] 데이터가 nutrition_db.json에 저장되었습니다.
모든 작업이 완료되었습니다: ['[완료] 수박']
